In [ ]:
!pip install trl
# 1. Force upgrade bitsandbytes to satisfy the new Hugging Face requirement
!pip install -U bitsandbytes>=0.46.1

# 2. Pull in standard tools without letting them touch or upgrade PyTorch
!pip install -q accelerate transformers scikit-learn

# 3. Pull in the remaining wrappers using safe isolation
!pip install -q --no-deps datasets trl peft huggingface_hub

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 630.8/630.8 kB 9.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 527.0/527.0 kB 16.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 47.6/47.6 MB 12.3 MB/s eta 0:00:00
  Attempting uninstall: pyarrow
    Found existing installation: pyarrow 18.1.0
    Uninstalling pyarrow-18.1.0:
      Successfully uninstalled pyarrow-18.1.0
  Attempting uninstall: datasets
    Found existing installation: datasets 4.0.0
    Uninstalling datasets-4.0.0:
      Successfully uninstalled datasets-4.0.0


In [ ]:
import os
import pandas as pd
import torch
from datasets import Dataset
from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    BitsAndBytesConfig
)
from peft import LoraConfig, get_peft_model
from trl import SFTConfig, SFTTrainer

# ==========================================
# 1. LOAD AND FORMAT YOUR FILTERED DATASET
# ==========================================

# Let's create dummy clinical data to simulate your filtered CSV
dummy_data = {
    'HR': [72, 110, 60, 125],
    'Temp': [98.6, 102.5, 97.0, 103.1],
    'SBP': [120, 85, 115, 80],
    'WBC': [8, 15, 6, 18],
    'Clinical_Note': [
        "Patient resting comfortably.",
        "Saan lenay mein masla hai.",
        "Routine checkup.",
        "Jism thanda par raha hai."
    ],
    'SepsisLabel': [0, 1, 0, 1]
}

df = pd.DataFrame(dummy_data)

# Convert Pandas DataFrame to Hugging Face Dataset
dataset = Dataset.from_pandas(df)

# We map the data manually! This guarantees no library crashes in the trainer.
def process_data(example):
    status = "CRITICAL" if example['SepsisLabel'] == 1 else "STABLE"
    text = f"""### Instruction:
Analyze the following ICU patient parameters for Sepsis risk.

### Input:
HR: {example['HR']} bpm
Temp: {example['Temp']} °F
SBP: {example['SBP']} mmHg
WBC: {example['WBC']} k
Note: {example['Clinical_Note']}

### Response:
[STATUS]: {status}. Based on clinical guidelines, these parameters indicate the patient is {status.lower()}.
"""
    return {"text": text}

# Apply the mapping directly
formatted_dataset = dataset.map(process_data)

# ==========================================
# 2. LOAD QUANTIZED GEMMA MODEL (4-BIT)
# ==========================================

model_id = "google/gemma-2-2b-it"

# Configure 4-bit quantization to fit on free GPUs
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16
)

tokenizer = AutoTokenizer.from_pretrained(model_id)
model = AutoModelForCausalLM.from_pretrained(
    model_id,
    quantization_config=bnb_config,
    device_map="auto"
)

# ==========================================
# 3. CONFIGURE LORA (PEFT)
# ==========================================

lora_config = LoraConfig(
    r=8,
    lora_alpha=16,
    target_modules=["q_proj", "o_proj", "k_proj", "v_proj", "gate_proj", "up_proj", "down_proj"],
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM"
)

model = get_peft_model(model, lora_config)

# ==========================================
# 4. INITIALIZE TRAINER
# ==========================================

training_args = SFTConfig(
    output_dir="./medgemma_output",
    per_device_train_batch_size=1,
    gradient_accumulation_steps=4,
    learning_rate=2e-4,
    max_steps=10,
    logging_steps=1,
    fp16=False,
    bf16=True,
    max_length=512,
)

trainer = SFTTrainer(
    model=model,
    train_dataset=formatted_dataset, # We pass the already-formatted dataset here
    args=training_args,
)

# ==========================================
# 5. RUN TRAINING & SAVE
# ==========================================
print("🚀 Starting Fine-Tuning...")
trainer.train()

# Save the small LoRA adapter weights
model.save_pretrained("./medgemma_adapter")
print("✅ Training complete! Weights saved to ./medgemma_adapter")

Map:   0%|          | 0/4 [00:00<?, ? examples/s]

Loading weights:   0%|          | 0/288 [00:00<?, ?it/s]

Adding EOS to train dataset:   0%|          | 0/4 [00:00<?, ? examples/s]

Tokenizing train dataset:   0%|          | 0/4 [00:00<?, ? examples/s]

The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'eos_token_id': 1}.


🚀 Starting Fine-Tuning...


Step,Training Loss
1,3.683859
2,2.924403
3,2.225206
4,1.726587
5,1.343039
6,1.054938
7,0.851511
8,0.707945
9,0.610382
10,0.552832


✅ Training complete! Weights saved to ./medgemma_adapter


In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM
from peft import PeftModel
import torch

# 1. Load the base Gemma model again
model_id = "google/gemma-2-2b-it"
tokenizer = AutoTokenizer.from_pretrained(model_id)
base_model = AutoModelForCausalLM.from_pretrained(
    model_id,
    torch_dtype=torch.bfloat16,
    device_map="auto"
)

# 2. Inject your newly trained Sepsis brain on top
trained_model = PeftModel.from_pretrained(base_model, "./medgemma_adapter")

# 3. Create a brand new fake patient
test_prompt = """### Instruction:
Analyze the following ICU patient parameters for Sepsis risk.

### Input:
HR: 130 bpm
Temp: 104.0 °F
SBP: 70 mmHg
WBC: 20 k
Note: Mareez ko tez bukhar hai aur saans lene me shadeed dushwari hai.

### Response:
"""

# 4. Generate the prediction
inputs = tokenizer(test_prompt, return_tensors="pt").to("cuda")
outputs = trained_model.generate(**inputs, max_new_tokens=100)

print(tokenizer.decode(outputs[0], skip_special_tokens=True))

`torch_dtype` is deprecated! Use `dtype` instead!


Loading weights:   0%|          | 0/288 [00:00<?, ?it/s]

### Instruction:
Analyze the following ICU patient parameters for Sepsis risk.

### Input:
HR: 130 bpm
Temp: 104.0 °F
SBP: 70 mmHg
WBC: 20 k
Note: Mareez ko tez bukhar hai aur saans lene me shadeed dushwari hai.

### Response:
[STATUS]: CRITICAL. Based on clinical guidelines, these parameters indicate critical condition.

